# 1A. Construcción del dataset: selección de artistas
- Fuente: *Make Music Equal* (Chartmetric), versión del 3/3/2026, licencia CC BY-SA 4.0. [mme_artist_info.csv](https://chartmetric-public.s3.us-west-2.amazonaws.com/make-music-equal/mme_artist_info.csv)
- Depuración: 936.420 registros, 172 duplicados (nombre + URL) eliminados, 936.248 artistas.
- Salida: listados de la cohorte 1 (artistas 1–1.400) y la cohorte 2 (1.401–7.000), ordenados por `cm_artist_rank`.

Nota: el rank de Make Music Equal (3/3/2026) y el de la API (23/3/2026) pueden diferir levemente; el rank solo se usa para seleccionar artistas.

In [15]:
# !pip install pandas numpy matplotlib seaborn scikit-learn jupyter
# !pip install matplotlib
#import sys
#!{sys.executable} -m pip install matplotlib seaborn

In [ ]:
import pandas as pd


# Carga del dataset

In [ ]:
df = pd.read_csv("mme_artist_info_DATASET.csv")

C:\Users\Silvana\AppData\Local\Temp\ipykernel_8968\2302022066.py:1: DtypeWarning: Columns (0: chartmetric_rank) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("mme_artist_info_DATASET.csv")


In [19]:
df.columns

Index(['chartmetric_rank', 'chartmetric_id', 'artist_name', 'chartmetric_url',
       'country_name', 'composition', 'career_stage', 'pronouns', 'genre',
       'data_date'],
      dtype='str')

In [20]:
df.shape

(936420, 10)

# Duplicados

In [ ]:
# --- 1. CONFIGURACIÓN Y KEY ÚNICA SEGURA ---
# Uso del ID: nunca es NaN y es la referencia de Chartmetric
df['unique_entity_key'] = df['chartmetric_id'].astype(str)

# --- 2. ANÁLISIS DE COLISIONES (Nombres iguales, diferentes URL, no son dupli) ---
# Agrupación por nombre y conteo de URLs distintas por nombre
analisis_urls = df.groupby('artist_name')['chartmetric_url'].nunique()
colisiones_identidad = analisis_urls[analisis_urls > 1]

# --- 3. IDENTIFICACIÓN DE DUPLICADOS TÉCNICOS ---
# Casos donde la combinación Nombre + URL se repite (deberían ser el mismo registro)
mask_duplicados_tecnicos = df.duplicated(subset=['unique_entity_key'], keep=False)
df_duplicados = df[mask_duplicados_tecnicos].copy()

# Separación dentro de estos duplicados técnicos:
# A. Filas Espejo: Son copias exactas en todas las columnas
filas_espejo = df_duplicados[df_duplicados.duplicated(keep='first')]

# B. Filas Discrepantes: Mismo artista/URL pero algo cambia en las otras columnas
# keep=False marca todas las filas con conflicto
discrepantes = df_duplicados[~df_duplicados.duplicated(subset=df.columns.difference(['unique_entity_key']), keep=False)]

# --- 4. REPORTE FINAL ---
print("=== INFORME DE CALIDAD DE DATOS ===")
print(f"1. Colisiones de nombre (mismo nombre diferente url): {len(colisiones_identidad)}")
print(f"2. Duplicados Técnicos (Mismo artista + URL): {df.duplicated('unique_entity_key').sum()}")
print(f"   -> Copias exactas (Espejos): {len(filas_espejo)}")
print(f"   -> Registros con datos contradictorios: {len(discrepantes)}")
print("-" * 35)

# Visualización de discrepancias si existen
if not discrepantes.empty:
    print("Muestra de registros con datos discrepantes (mismo artista, distintos valores):")
    print(discrepantes.sort_values(by='unique_entity_key').head(10))

=== INFORME DE CALIDAD DE DATOS ===
1. Colisiones de nombre (mismo nombre diferente url): 31943
2. Duplicados Técnicos (Mismo artista + URL): 172
   -> Copias exactas (Espejos): 172
   -> Registros con datos contradictorios: 0
-----------------------------------


In [ ]:

total_antes = len(df)
df_sindupli = df.drop_duplicates(keep='first').copy()
eliminados = total_antes - len(df_sindupli)

# Verificación final mejorada
print(f"=== RESULTADO DE LA LIMPIEZA ===")
print(f"- Registros originales: {total_antes}")
print(f"- Espejos exactos eliminados: {eliminados}")
print(f"- Artistas con nombre NaN conservados: {df_sindupli['artist_name'].isna().sum()}")
print(f"- Tamaño final del dataset: {len(df_sindupli)} filas.")

=== RESULTADO DE LA LIMPIEZA ===
- Registros originales: 936420
- Espejos exactos eliminados: 172
- Artistas con nombre NaN conservados: 8
- Tamaño final del dataset: 936248 filas.


# Valores faltantes en `chartmetric_rank`

In [28]:
# Conversión de la columna a numérica; lo que no sea número queda NaN
df_sindupli['chartmetric_rank'] = pd.to_numeric(df_sindupli['chartmetric_rank'], errors='coerce')

In [29]:
# Orden de menor a mayor rank (el rank 1 arriba)
df_sindupli = df_sindupli.sort_values(by='chartmetric_rank', ascending=True).reset_index(drop=True)

In [31]:
# Cantidad de NaN después de la conversión
nulos_count = df_sindupli['chartmetric_rank'].isna().sum()
total_filas = len(df_sindupli)

print(f"Total de registros: {total_filas}")
print(f"Registros sin Rank (NaN): {nulos_count}")
print(f"Porcentaje de nulos: {(nulos_count / total_filas) * 100:.2f}%")

Total de registros: 936248
Registros sin Rank (NaN): 4
Porcentaje de nulos: 0.00%


In [32]:
# Filas donde el rank es NaN
df_nulos = df_sindupli[df_sindupli['chartmetric_rank'].isna()]

print(f"--- DETALLE DE LOS {len(df_nulos)} REGISTROS CON VALOR NaN ---")
# Columnas principales para identificar a los artistas
print(df_nulos[['artist_name', 'chartmetric_rank', 'chartmetric_id', 'chartmetric_url']])

--- DETALLE DE LOS 4 REGISTROS CON VALOR NaN ---
         artist_name  chartmetric_rank  chartmetric_id  \
936244  Jabbawockeez               NaN          573449   
936245  Jill & Julia               NaN          342683   
936246     Ammi Boyz               NaN          887992   
936247  Steeven Savy               NaN         3939422   

                                   chartmetric_url  
936244   https://app.chartmetric.com/artist/573449  
936245   https://app.chartmetric.com/artist/342683  
936246   https://app.chartmetric.com/artist/887992  
936247  https://app.chartmetric.com/artist/3939422  


## Eliminación de 4 NaN en rank

In [33]:
# 1. Conteo previo al borrado, para la documentación
total_antes = len(df_sindupli)
nulos_a_borrar = df_sindupli['chartmetric_rank'].isna().sum()

# 2. Eliminación de las filas donde chartmetric_rank es NaN
df_sindupli = df_sindupli.dropna(subset=['chartmetric_rank']).copy()

# 3. Orden por rank y reseteo del índice
# Esto asegura que el registro físico 0 sea el Rank 1, el 1000 sea el 1001, etc.
df_sindupli = df_sindupli.sort_values(by='chartmetric_rank', ascending=True).reset_index(drop=True)

# 4. Verificación final
total_despues = len(df_sindupli)
print(f"Registros iniciales: {total_antes}")
print(f"Registros eliminados (NaN): {nulos_a_borrar}")
print(f"Registros finales: {total_despues}")
print(f"--- Dataset ordenado y listo para segmentación ---")

Registros iniciales: 936248
Registros eliminados (NaN): 4
Registros finales: 936244
--- Dataset ordenado y listo para segmentación ---


# Cohorte 1: `df_1400`

In [ ]:
# 1. Recorte de los primeros 1400 registros (Slicing de Python)
# Como df_sindupli ya está ordenado por rank y con index reseteado,
# esto garantiza los 1400 mejores artistas de la lista.
df_1400 = df_sindupli.iloc[:1400].copy()

# 2. Columnas de interés para la auditoría de estratos
estratos = ['country_name', 'composition', 'career_stage', 'genre', 'pronouns']

print("--- AUDITORÍA DE VALORES ÚNICOS (BLOQUE 1-1400) ---")

for col in estratos:
    # Valores únicos
    uniques = df_1400[col].unique()
    # Cantidad
    count = len(uniques)
    
    print(f"\nColumna: {col} ({count} categorías)")
    print(uniques)
    print("-" * 40)

# 3. Cantidad de combinaciones únicas de estratos en este recorte
combinaciones = df_1400[estratos].drop_duplicates().shape[0]
print(f"\nTotal de 'perfiles' únicos (combinaciones de estratos): {combinaciones}")

# --- EXPORTACIÓN PARA LA API ---

# 4. Columnas necesarias para el proceso de descarga
columnas_interes = ['chartmetric_rank', 'chartmetric_id', 'artist_name', 'chartmetric_url']
df_1400_api = df_1400[columnas_interes]

# 5. Exportación a un archivo .txt
# sep='\t' (tabulador) para evitar conflictos con comas en nombres de artistas
nombre_archivo = 'df_1400_api.txt'
df_1400_api.to_csv(nombre_archivo, sep='\t', index=False, encoding='utf-8')

print(f"\nArchivo '{nombre_archivo}' creado con éxito.")
print(f"Rango de Rank exportado: {df_1400_api['chartmetric_rank'].min()} a {df_1400_api['chartmetric_rank'].max()}")

--- AUDITORÍA DE VALORES ÚNICOS (BLOQUE 1-1400) ---

Columna: country_name (58 categorías)
<StringArray>
[                         'Puerto Rico',
                        'United States',
                               'Canada',
                             'Barbados',
                       'United Kingdom',
                             'Colombia',
                               'France',
                                'India',
                               'Sweden',
                          'South Korea',
                            'Australia',
                              'Jamaica',
                               'Mexico',
                         'South Africa',
                              'Senegal',
                                'Spain',
                          'Netherlands',
                            'Argentina',
                              'Nigeria',
                               'Norway',
                               'Brazil',
                              'Ire

# Cohorte 2: `df_1401_7000`

In [ ]:
# 1. Recorte del registro 1400 al 7000 (Slicing de Python)
# Esto toma las filas desde el índice 1400 hasta el 5999.
df_1401_7000 = df_sindupli.iloc[1400:7000].copy()

# 2. Columnas de interés para la auditoría de estratos
estratos = ['country_name', 'composition', 'career_stage', 'genre', 'pronouns']

print("--- AUDITORÍA DE VALORES ÚNICOS (BLOQUE 1401-7000) ---")

for col in estratos:
    # Valores únicos
    uniques = df_1401_7000[col].unique()
    # Cantidad
    count = len(uniques)
    
    print(f"\nColumna: {col} ({count} categorías)")
    print(uniques)
    print("-" * 40)

# 3. Cantidad de combinaciones únicas de estratos en este recorte
combinaciones = df_1401_7000[estratos].drop_duplicates().shape[0]
print(f"\nTotal de 'perfiles' únicos (combinaciones de estratos) en este bloque: {combinaciones}")

# --- EXPORTACIÓN PARA LA API ---

# 4. Columnas necesarias para el proceso de descarga
columnas_interes = ['chartmetric_rank', 'chartmetric_id', 'artist_name', 'chartmetric_url']
df_1401_7000_api = df_1401_7000[columnas_interes]

# 5. Exportación a un archivo .txt
nombre_archivo = 'df_1401_7000_api.txt'
df_1401_7000_api.to_csv(nombre_archivo, sep='\t', index=False, encoding='utf-8')

print(f"\nArchivo '{nombre_archivo}' creado con éxito.")
print(f"Rango de Rank exportado: {df_1401_7000_api['chartmetric_rank'].min()} a {df_1401_7000_api['chartmetric_rank'].max()}")

--- AUDITORÍA DE VALORES ÚNICOS (BLOQUE 1401-7000) ---

Columna: country_name (106 categorías)
<StringArray>
[             'Brazil',       'United States',      'United Kingdom',
              'Mexico',           'Argentina',         'South Korea',
             'Jamaica',           'Australia',        'South Africa',
         'Puerto Rico',
 ...
           'Sri Lanka',               'Samoa',             'Senegal',
          'Madagascar', 'Trinidad and Tobago',               'Gabon',
              'Serbia',             'Hungary',            'Portugal',
            'Barbados']
Length: 106, dtype: str
----------------------------------------

Columna: composition (3 categorías)
<StringArray>
['solo', 'group', 'unknown']
Length: 3, dtype: str
----------------------------------------

Columna: career_stage (5 categorías)
<StringArray>
['superstar', 'legendary', 'mainstream', 'mid-level', 'developing']
Length: 5, dtype: str
----------------------------------------

Columna: genre (324 catego

In [ ]:
print("--- VERIFICACIÓN DE TRANSICIÓN ---")
print(f"BLOQUE 1400 (Última fila): ID {df_1400['chartmetric_id'].iloc[-1]} | Rank: {df_1400['chartmetric_rank'].iloc[-1]}")
print(f"BLOQUE 1401_7000 (Primera fila): ID {df_1401_7000['chartmetric_id'].iloc[0]} | Rank: {df_1401_7000['chartmetric_rank'].iloc[0]}")

# Verificación de saltos grandes en el ranking
salto = df_1401_7000['chartmetric_rank'].iloc[0] - df_1400['chartmetric_rank'].iloc[-1]
print(f"Distancia de ranking entre bloques: {salto}")

--- VERIFICACIÓN DE TRANSICIÓN ---
BLOQUE 1400 (Última fila): ID 3917158 | Rank: 1704.0
BLOQUE 1401_7000 (Primera fila): ID 1103173 | Rank: 1705.0
Distancia de ranking entre bloques: 1.0
